In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

spy = yf.download("SPY", start="2015-01-01")["Close"]
qqq = yf.download("QQQ", start="2015-01-01")["Close"]
smh = yf.download("SMH", start="2015-01-01")["Close"]

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


We choose SPY, QQQ, and SMH as the stocks to construct a portfolio with. 

In [2]:
prices = pd.concat([spy, qqq, smh], axis=1)
prices.columns = ["SPY", "QQQ", "SMH"]

prices = prices.dropna()
prices.head()

,SPY,QQQ,SMH
2015-01-02,170.125015,94.665070,24.292725
2015-01-05,167.052612,93.276459,23.860281
2015-01-06,165.479141,92.025780,23.298546
2015-01-07,167.541199,93.212074,23.601702
2015-01-08,170.514236,94.996147,24.190186


Create a new dataframe based on the prices dataframe containing the daily return for each ticker.

In [3]:
returns = prices.pct_change().dropna()
returns.head()

,SPY,QQQ,SMH
2015-01-05,-0.018060,-0.014669,-0.017801
2015-01-06,-0.009419,-0.013408,-0.023543
2015-01-07,0.012461,0.012891,0.013012
2015-01-08,0.017745,0.019140,0.024934
2015-01-09,-0.008013,-0.006583,-0.002949


We will construct our portfolio as 40% SPY, 40% QQQ, and 20% SMH. From these weights, we append the portfolio return to our existing dataframe. 

In [7]:
allocation = np.array([0.4,0.4,0.2])
portfolio_returns = returns[["SPY", "QQQ","SMH"]] @ allocation
returns["Portfolio"] = portfolio_returns
returns.head()

,SPY,QQQ,SMH,Portfolio
2015-01-05,-0.018060,-0.014669,-0.017801,-0.016652
2015-01-06,-0.009419,-0.013408,-0.023543,-0.013839
2015-01-07,0.012461,0.012891,0.013012,0.012743
2015-01-08,0.017745,0.019140,0.024934,0.019741
2015-01-09,-0.008013,-0.006583,-0.002949,-0.006428


Sanity check that our portfolio return is as expected.

In [8]:
-0.009419*.4+-0.013408*0.4+-0.023543*0.2

-0.013839400000000002

Correlation matrix. This portfolio has minimal diversification as the assets held in each ETF are often held amongst all of the funds and concentrated in tech. 

In [10]:
returns.corr()

,SPY,QQQ,SMH,Portfolio
SPY,1.000000,0.934127,0.813464,0.959628
QQQ,0.934127,1.000000,0.872927,0.982490
SMH,0.813464,0.872927,1.000000,0.927626
Portfolio,0.959628,0.982490,0.927626,1.000000


A few basic risk measures. 

In [13]:
portfolio_vol = returns["Portfolio"].std()
VaR_95 = np.percentile(returns["Portfolio"], 5)
ES_95 = returns[returns["Portfolio"] <= VaR_95]["Portfolio"].mean()

print("Portfolio Volatility:", portfolio_vol)
print("Portfolio VaR (95%):", VaR_95) 
print("Portfolio ES (95%):", ES_95)

Portfolio Volatility: 0.013362577806298803
Portfolio VaR (95%): -0.020808506502489256
Portfolio ES (95%): -0.03177564157272604


The portfolio partially reduces SMH risk but remains exposed to tech-driven downside risk, with tail risk closer to QQQ than SPY.

In [14]:
comparison = pd.DataFrame({
    "VaR_95": returns[["SPY","QQQ","SMH","Portfolio"]].apply(lambda x: np.percentile(x, 5)),
    "Volatility": returns[["SPY","QQQ","SMH","Portfolio"]].std()
})

comparison

,VaR_95,Volatility
SPY,-0.016666,0.011128
QQQ,-0.022012,0.013772
SMH,-0.030827,0.019830
Portfolio,-0.020809,0.013363


In [16]:
returns["Portfolio_Breach"] = returns["Portfolio"] < VaR_95

portfolio_breach_rate = returns["Portfolio_Breach"].mean()

print("Portfolio VaR:", VaR_95)
print("Observed breach rate:", portfolio_breach_rate)
print("Expected breach rate: 0.05")

Portfolio VaR: -0.020808506502489256
Observed breach rate: 0.050296891372685996
Expected breach rate: 0.05


## Interpretation 

The portfolio reduces the outlier downside risk of SMH, but remains closer to QQQ in both volatility and VaR, indicating strong exposure to tech sector risk factors. 

This is expected as SMH is largely a subset of equities within QQQ and thus our diversification benefits are limited. 